# Endpoint quickstart

Same recipe as `examples/endpoint_quickstart.py`: vLLM serving `Qwen/Qwen3-0.6B` on one L40S, behind token auth. The notebook creates the endpoint, waits for its URL, waits for the model, sends one chat request, and deletes the endpoint.

**Before you start**

- Run `uv sync` in the repo root, then start Jupyter from the same environment, for example `uv run --with jupyter jupyter lab`.
- You need a Nebius CLI profile (`nebius profile create`). The active profile supplies the project. The region comes from the profile or the project.
- You need quota for one on-demand L40S in `eu-north1`. For another region, change `gpu=`. The alias table is in `docs/reference.md`.

**An endpoint bills from create until delete.** Interrupting a cell, a local `WaitTimeout`, and restarting the kernel all leave it running. If anything goes wrong after the create cell, run the last cell, **Delete**.

Jupyter already runs an event loop, so every call that talks to Nebius uses the awaited twin (`create_async`, `wait_async`, …). A synchronous `create()` in this kernel raises `ValidationError` and names the twin to use.

In [5]:
import asyncio
import json
import urllib.error
import urllib.request

import nebius_serverless as ns

MODEL = "Qwen/Qwen3-0.6B"

## Recipe

An `Endpoint` is a local recipe. Constructing it does not contact Nebius.

`command` overrides the image entrypoint. Omit it to keep the entrypoint. With `auth="token"` and no `token=`, a random bearer token is generated on create and is available as `service.token`.

In [9]:
endpoint = ns.Endpoint(
    [
        "bash",
        "-c",
        "python3 -m vllm.entrypoints.openai.api_server "
        f"--model {MODEL} --host 0.0.0.0 --port 8000",
    ],
    image="vllm/vllm-openai:v0.19.1",
    gpu="h100",
    port=8000,
    auth="token",
    disk="500Gi",
    name="qwen-tiny",
)

{
    "name": endpoint.name,
    "command": list(endpoint.command),
    "image": endpoint.image,
    "gpu": endpoint.gpu,
    "port": endpoint.port,
    "auth": endpoint.auth,
}

{'name': 'qwen-tiny',
 'command': ['bash',
  '-c',
  'python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000'],
 'image': 'vllm/vllm-openai:v0.19.1',
 'gpu': 'h100',
 'port': 8000,
 'auth': 'token'}

## Dry run

This resolves the project, region, platform, and preset, and returns the create request. It does not create an endpoint. The token is shown as `"generated"`, never as a value.

In [10]:
await endpoint.create_async(dry_run=True)

{'image': 'vllm/vllm-openai:v0.19.1',
 'command': ['bash',
  '-c',
  'python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000'],
 'args': "-c 'python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000'",
 'port': 8000,
 'project': 'project-e00chatepr006ppncgsanv',
 'region': 'eu-north1',
 'platform': 'gpu-h100-sxm',
 'preset': '1gpu-16vcpu-200gb',
 'pricing': 'on-demand',
 'auth': 'token',
 'token': 'generated',
 'env': [],
 'name': 'qwen-tiny',
 'disk': '500Gi',
 'shm': '16Gi'}

## Create

`create_async` returns a `Service` as soon as Nebius accepts the endpoint and its id is known. It does not wait for the URL. **This cell starts a billable endpoint.**

Save `service.id` so you can find the endpoint again. `service.token` exists only on this object. A later `endpoints.get` does not return it.

In [11]:
service = await endpoint.create_async()

endpoint_id = service.id
endpoint_id, service.status

Submitted endpoint aiendpoint-e00q1azb6saq2fjzv4 qwen-tiny


('aiendpoint-e00q1azb6saq2fjzv4', None)

## Wait for the URL

This polls about every 5 seconds until the state is `RUNNING` with a managed `https://` URL, or until it is `ERROR` or `STOPPED`. Pulling the vLLM image takes a few minutes.

`timeout=` limits only the local wait. When it expires, the endpoint keeps running and billing. Re-run the cell to keep waiting.

In [12]:
try:
    await service.wait_async(timeout=20 * 60)
except ns.WaitTimeout as exc:
    print(f"Stopped waiting; endpoint {exc.run_id} is still {service.status}.")
    print("Re-run this cell to keep waiting, or delete it in the last cell.")
else:
    service.display()

aiendpoint-e00q1azb6saq2fjzv4 PROVISIONING · 0.2s
aiendpoint-e00q1azb6saq2fjzv4 STARTING · 57.9s
aiendpoint-e00q1azb6saq2fjzv4 IMAGE_PULLING · 94.9s
✓ aiendpoint-e00q1azb6saq2fjzv4 RUNNING · 229.3s
  https://port8000-gc5hrgvr05pb555.tunnel.applications.eu-north1.nebius.cloud


aiendpoint-e00q1azb6saq2fjzv4  qwen-tiny
  RUNNING

  url       https://port8000-gc5hrgvr05pb555.tunnel.applications.eu-north1.nebius.cloud

  image     vllm/vllm-openai:v0.19.1
  command   bash
  platform  gpu-h100-sxm
  preset    1gpu-16vcpu-200gb
  disk      500 GiB NETWORK_SSD
  shm       16 GiB
  subnet    vpcsubnet-e00gk7s3thae0shr02

  created   2026-11-13 11:46:28Z

  instance  computeinstance-e00eeqts6g9h4k92zz  10.0.0.81  RUNNING


## Wait for the model

`RUNNING` means the container is up. vLLM may still be loading weights, so poll the application's own `/v1/models` route before sending traffic.

In [13]:
def request(path, body=None):
    headers = {"Authorization": f"Bearer {service.token}"}
    data = None
    if body is not None:
        headers["Content-Type"] = "application/json"
        data = json.dumps(body).encode()
    return urllib.request.Request(
        service.url.rstrip("/") + path, data=data, headers=headers
    )


def fetch(path, body=None, timeout=10):
    with urllib.request.urlopen(request(path, body), timeout=timeout) as r:
        return r.status, json.load(r)


async def wait_for_model(limit=15 * 60):
    loop = asyncio.get_running_loop()
    deadline = loop.time() + limit
    while True:
        try:
            status, payload = await asyncio.to_thread(fetch, "/v1/models")
            if status == 200:
                return payload
        except (urllib.error.URLError, TimeoutError):
            pass
        if loop.time() > deadline:
            raise TimeoutError(f"{service.url} did not answer /v1/models")
        await asyncio.sleep(10)


assert service.url, f"No URL yet: {service.status}"
models = await wait_for_model()
[model["id"] for model in models["data"]]

['Qwen/Qwen3-0.6B']

## Call the model

This client does not wrap inference. Send HTTP yourself to `service.url` with the bearer token. vLLM serves the OpenAI-compatible API.

In [15]:
status, payload = await asyncio.to_thread(
    fetch,
    "/v1/chat/completions",
    {
        "model": MODEL,
        "messages": [{"role": "user", "content": "Say hello in five words."}],
        "max_tokens": 256,
    },
    60,
)
print(payload["choices"][0]["message"]["content"])

<think>
Okay, the user wants me to say hello in five words. Let me think. First, I need to make sure it's exactly five words. Maybe start with "Hi!" and then add four more. But how to keep it concise. "Hi!" is one word, so maybe "Hi! How are you?" That's four words. Wait, but the user wants five words. Let me count: "Hi! How are you?" That's five words. Yes, that works. But maybe the user just wants a simple greeting. "Hello!" is shorter. But the user specified five words. Let me check again. "Hi! How are you?" is five words. Alternatively, "Hello!" is four, so adding another word. Maybe "Hello!" plus a second word. Wait, but the user wants five. Let me think of other possibilities. "Hi! How are you?" is five. Or "Hi, how are you?" That's also five. Either way, but the user might prefer a more direct answer. Since the example response uses "Hi! How are you?" I think that's acceptable.
</think>

Hi! How are you?


## Reload by id

`endpoints.get_async` starts from an id string. Use it after a kernel restart, in another notebook, or for an endpoint created from the CLI. It fetches once and never creates an endpoint. The reloaded `Service` has the URL but no token.

In [16]:
loaded = await ns.endpoints.get_async(endpoint_id)

loaded.id, loaded.status, loaded.url, loaded.token

aiendpoint-e00q1azb6saq2fjzv4 RUNNING


('aiendpoint-e00q1azb6saq2fjzv4',
 'RUNNING',
 'https://port8000-gc5hrgvr05pb555.tunnel.applications.eu-north1.nebius.cloud',
 None)

## Delete

`delete_async` is the only call that stops billing. It returns once the request is accepted. A later `get_async` shows `DELETING` or reports that the id is missing.

After a kernel restart, `service` is gone. Run `await ns.endpoints.get_async("<endpoint id>")` and then `delete_async()` on the result.

In [ ]:
await service.delete_async()

try:
    await service.get_async()
except ns.Error as exc:
    print(f"{endpoint_id} is gone: {exc}")
else:
    print(service.status)